# Notebook 2: Data Preprocessing & Feature Engineering
Prepares data for ARIMA and Prophet modelling pipelines.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
warnings.filterwarnings('ignore')

df_raw = pd.read_csv('../data/raw/synthetic_sales_data.csv', parse_dates=['date'])
df_agg = pd.read_csv('../data/raw/aggregated_monthly_sales.csv', parse_dates=['date'])
print('Loaded. Shapes:', df_raw.shape, df_agg.shape)

## 1. Handling Missing Values

In [ ]:
# Lag features are NaN for first 3 months of each series — expected, not a data quality issue
print('Missing values before treatment:')
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])

# Forward fill lag features per product-region group
lag_cols = ['lag_1_units', 'lag_3_units', 'rolling_avg_3']
for col in lag_cols:
    df_raw[col] = df_raw.groupby(['product_name', 'region'])[col].transform(lambda x: x.fillna(x.median()))

print('\nMissing values after treatment:')
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])
print('No missing values remain.')

## 2. Outlier Detection (IQR Method)

In [ ]:
def detect_outliers_iqr(series, k=1.5):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - k * IQR
    upper = Q3 + k * IQR
    return (series < lower) | (series > upper)

outlier_mask = detect_outliers_iqr(df_raw['units_sold'])
print(f'Outliers detected: {outlier_mask.sum()} rows ({outlier_mask.mean()*100:.2f}%)')

# Cap outliers (winsorize) instead of removing — preserves time series continuity
Q1 = df_raw['units_sold'].quantile(0.25)
Q3 = df_raw['units_sold'].quantile(0.75)
IQR = Q3 - Q1
df_raw['units_sold_clean'] = df_raw['units_sold'].clip(lower=Q1 - 1.5*IQR, upper=Q3 + 1.5*IQR)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.boxplot(df_raw['units_sold'], vert=True)
ax1.set_title('Before Outlier Treatment')
ax2.boxplot(df_raw['units_sold_clean'], vert=True)
ax2.set_title('After Winsorization')
plt.tight_layout()
plt.savefig('../reports/figures/08_outlier_treatment.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Stationarity Test (Augmented Dickey-Fuller)

In [ ]:
# Test stationarity for each product's aggregate series
print(f'{'Product':<20} {'ADF Statistic':>15} {'p-value':>10} {'Stationary?':>12}')
print('-' * 60)

results_stationarity = {}
for prod in df_agg['product_name'].unique():
    series = df_agg[df_agg['product_name'] == prod].set_index('date')['total_units_sold']
    result = adfuller(series, autolag='AIC')
    stationary = result[1] < 0.05
    results_stationarity[prod] = stationary
    print(f'{prod:<20} {result[0]:>15.4f} {result[1]:>10.4f} {"Yes" if stationary else "No":>12}')

## 4. ACF / PACF Plots (for ARIMA order selection)

In [ ]:
# Pick one representative product: Escitab-10 (highest demand + clear seasonality)
sample = df_agg[df_agg['product_name'] == 'Escitab-10'].set_index('date')['total_units_sold']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(sample, lags=20, ax=ax1, title='ACF — Escitab-10')
plot_pacf(sample, lags=20, ax=ax2, title='PACF — Escitab-10', method='ywm')
plt.tight_layout()
plt.savefig('../reports/figures/09_acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()
print('ACF/PACF: Use these to set p and q for ARIMA(p,d,q).')
print('Rule: PACF cuts off at lag p → AR order; ACF cuts off at lag q → MA order')

## 5. Additional Feature Engineering

In [ ]:
def engineer_features(df):
    df = df.copy()
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['is_q4'] = (df['month'].isin([10, 11, 12])).astype(int)
    df['is_winter'] = (df['month'].isin([11, 12, 1, 2])).astype(int)
    df['is_diwali_month'] = (df['month'] == 10).astype(int)
    df['year_index'] = df['year'] - df['year'].min()  # 0, 1, 2
    return df

df_raw = engineer_features(df_raw)
df_agg = engineer_features(df_agg)

print('New features added:')
new_cols = ['month_sin', 'month_cos', 'is_q4', 'is_winter', 'is_diwali_month', 'year_index']
print(df_raw[new_cols].head(6))

## 6. Train/Test Split

In [ ]:
# Standard practice: use last 6 months as test set
CUTOFF = '2024-06-01'

train_agg = df_agg[df_agg['date'] < CUTOFF]
test_agg  = df_agg[df_agg['date'] >= CUTOFF]

print(f'Train set: {train_agg["date"].min().date()} → {train_agg["date"].max().date()} ({len(train_agg)} rows)')
print(f'Test set:  {test_agg["date"].min().date()}  → {test_agg["date"].max().date()} ({len(test_agg)} rows)')

# Save splits
import os; os.makedirs('../data/processed', exist_ok=True)
df_raw.to_csv('../data/processed/featured_sales_data.csv', index=False)
df_agg.to_csv('../data/processed/featured_aggregated_data.csv', index=False)
train_agg.to_csv('../data/processed/train_agg.csv', index=False)
test_agg.to_csv('../data/processed/test_agg.csv', index=False)

print('\nAll processed files saved to data/processed/')